In [ ]:
# 1-3. Context, loading, quality
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
o = pd.read_csv("orders.csv")
i = pd.read_csv("order_items.csv")
p = pd.read_csv("products.csv")
for t in [o, i, p]: print(t.shape, t.isna().sum().to_dict())
print(o.duplicated().sum(), o.status.value_counts().to_dict())
print(o.channel.unique(), p.category.unique())
i.describe()

In [ ]:
# 4. Anomalies
d = o.order_date[o.order_date.str.contains("/")]
print(len(d), (d.str[:2].astype(int) > 12).sum(), (d.str[3:5].astype(int) > 12).sum())
i = i.merge(p[["product_id", "list_price"]], how="left")
i["ratio"] = i.unit_price / i.list_price
i["implied"] = i.line_total / i.unit_price
i[(i.ratio > 2) | (i.quantity >= 50) | i.unit_price.isna()]

In [ ]:
# 5. Transformation
oc = o.drop_duplicates().copy()
oc["order_date"] = pd.to_datetime(oc.order_date, format="ISO8601", errors="coerce").fillna(pd.to_datetime(oc.order_date, format="%d/%m/%Y", errors="coerce"))
oc["channel"] = oc.channel.str.strip().str.title()
oc["customer_id"] = oc.customer_id.fillna("GUEST")
p["category"] = p.category.str.strip().str.title()
i["price_raw"], i["qty_raw"] = i.unit_price, i.quantity
i["fix"] = np.select([i.ratio > 50, i.unit_price.isna(), i.quantity == 999], ["cents", "no price", "999 qty"], "")
i.loc[i.fix == "cents", "unit_price"] /= 100
i["unit_price"] = i.unit_price.fillna(i.line_total / i.quantity)
i.loc[i.fix == "999 qty", "quantity"] = i.implied
print(oc.order_date.isna().sum(), oc.channel.unique(), p.category.unique())
i.fix.value_counts()

In [ ]:
# 5. Reconciliation
c = i.drop(columns=["list_price", "ratio", "implied"]).merge(oc, how="left", validate="m:1")
c = c.merge(p[["product_id", "product_name", "category"]], how="left", validate="m:1")
c["category"] = c.category.fillna("Unknown")
c["exception"] = np.select([c.status.isna(), c.category == "Unknown", (c.quantity * c.unit_price - c.line_total).abs() > 0.01], ["no order", "unknown product", "total mismatch"], "")
c["in_revenue"] = c.status == "completed"
c.exception.value_counts()

In [ ]:
# 6. Business analysis
r = c[c.in_revenue]
ov = r.groupby(["order_id", "channel"], as_index=False).line_total.sum()
print(r.line_total.sum().round(2), len(ov), ov.line_total.mean().round(2))
display(r.groupby("category").line_total.sum().sort_values())
display(r.groupby("product_name").line_total.sum().nlargest(5))
m = r.groupby(r.order_date.dt.to_period("M")).line_total.sum()
display(pd.concat([m, m.pct_change()], axis=1).round(2))
r[r.order_date.dt.month == 6].groupby("channel").line_total.sum()

In [ ]:
# 6. Outliers (IQR on log10)
lg = np.log10(ov.line_total)
q1, q3 = lg.quantile([0.25, 0.75])
lim = q3 + 1.5 * (q3 - q1)
ov["outlier"] = lg > lim
print(10 ** lim)
ov.groupby("channel").outlier.agg(["sum", "count"])

In [ ]:
# 7. Visualizations
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
m.plot(ax=ax[0], marker="o", title="Monthly revenue")
sns.boxplot(data=ov, x="channel", y="line_total", log_scale=True, ax=ax[1])
plt.show()

In [ ]:
# 8. Validation
print(i.merge(o, on="order_id").line_total.sum().round(2), i.line_total.sum().round(2))
print(len(i), len(c), r.groupby("category").line_total.sum().sum().round(2))
c.groupby(c.status.fillna("no order")).line_total.sum()

In [ ]:
# 9. Deliverables
c.to_csv("clean_sales.csv", index=False)
rep = c[(c.fix != "") | (c.exception != "") | (c.in_revenue == False)]
rep.to_csv("exceptions_report.csv", index=False)
len(rep)

## 9. Conclusions
- Data: removed 20 duplicate orders (a naive join adds $4,911.90), fixed 127 day-first dates and 26 lines with evidence. Unresolved: 6 total mismatches, 8 P025 lines, 5 lines without an order.
- Business: revenue $433,779.55 from 1,328 orders, AOV $326.64. Bedding leads. June +50% came from 3 B2B orders; November is the peak.
- Outliers: 36 orders, all B2B: a segment, not errors.
- Next: one date format, prices in dollars, add P025, report B2B apart.
- Final notes: synthetic files generated with AI before recording.